# InsightVLM — NB4v2: Qwen2-VL-2B + QLoRA on version-2 charts

NB4 retrained on the clean v2 renders (no label-dependent marks). Same model, 4,000 training samples, one epoch, same prompt and the same 300-chart evaluation sample. Output: `nb4_results.json`, `nb4_vlm_predictions.csv`.

In [ ]:
!pip -q install -U "transformers>=4.45" accelerate peft trl bitsandbytes qwen-vl-utils
print("INSTALL DONE -- now RESTART THE KERNEL, then run the cells below.")

In [ ]:
import os, sys, json, time, re, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset as TorchDS

DATA_ROOT = Path("/kaggle/input/datasets/asifxzaman/xxxyyy/dataset/dataset")
DATA = DATA_ROOT if (DATA_ROOT / "records.jsonl").exists() else \
       list(Path("/kaggle/input").glob("**/records.jsonl"))[0].parent
print("dataset:", DATA)
meta = pd.read_parquet(DATA / "tables" / "metadata.parquet")
te = meta[meta.split == "test"].reset_index(drop=True)
_ci = json.loads((DATA / "class_index.json").read_text())["label_to_id"]
CLASSES = sorted(_ci, key=_ci.get)
print(f"{len(meta):,} samples | {len(CLASSES)} classes | test {len(te):,}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
RENDER_SRC = '"""\n06_render.py — Render each record to a chart image with pixel-valid grounding boxes.\n\nTwo rendering rules that the notebook prototype got wrong, and which matter:\n\n  1. NO bbox_inches="tight". Tight-cropping recomputes the canvas *after* the\n     axes transform has been read, so every box drawn from ax.transData is\n     shifted relative to the file that actually gets saved. The canvas here is\n     fixed, so display coordinates and image pixels are the same thing.\n  2. Boxes are emitted in image pixel space with the y axis flipped, because\n     matplotlib\'s origin is bottom-left and an image\'s is top-left.\n\nEach image carries evidence boxes marking the pixels that support the finding:\nthe outlier band, the fitted trend, the changepoint, the anomalous points, the\ndominant wedge. That grounding is what separates this from chart-QA datasets.\n"""\nimport argparse\nimport json\nimport os\nimport sys\nimport warnings\nimport zlib\nfrom collections import Counter\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt          # noqa: E402\nimport numpy as np                       # noqa: E402\nfrom matplotlib.gridspec import GridSpec  # noqa: E402\nfrom tqdm import tqdm                    # noqa: E402\n\nwarnings.filterwarnings("ignore")\n\nROOT = Path(__file__).resolve().parent.parent\nCACHE = ROOT / "cache"\nDATA = ROOT / "dataset"\n# env, not globals set in main(): spawned worker processes re-import this module\nIMG = Path(os.environ.get("RENDER_OUT", DATA / "images"))\n\nW_PX, H_PX, DPI = 768, 576, 100\nFIGSIZE = (W_PX / DPI, H_PX / DPI)\n\nSTYLES = [\n    ("seaborn-v0_8-whitegrid", "tab10"), ("seaborn-v0_8-darkgrid", "Set2"),\n    ("ggplot", "tab10"), ("bmh", "Dark2"), ("fivethirtyeight", "tab10"),\n    ("seaborn-v0_8-ticks", "colorblind"), ("default", "tab10"),\n    ("seaborn-v0_8-muted", "Set1"),\n]\n\n\n# --clean: draw no label-dependent marks. The trend line (only on increasing/\n# decreasing), changepoint line and anomaly circles each reveal the class, so a\n# model can read the annotation instead of the data. Boxes are still emitted.\nCLEAN = os.environ.get("RENDER_CLEAN") == "1"\n# --randstyle: training-only domain randomisation (fonts, line/marker sizes, colours,\n# grids, spines, backgrounds, titles). The data and the chart type are untouched, so\n# the label is too. Its own seed, so the same chart gets a different look.\nRANDSTYLE = os.environ.get("RENDER_RANDSTYLE") == "1"\nRAND_CMAPS = ["tab10", "Set1", "Set2", "Dark2", "Paired", "viridis", "plasma", "cividis", "Accent"]\n\n\ndef _rand_rc(rr):\n    import colorsys\n    tint = lambda: colorsys.hsv_to_rgb(rr.uniform(0, 1), rr.uniform(0, 0.12), rr.uniform(0.93, 1.0))  # noqa: E731\n    return {\n        "font.family": str(rr.choice(["DejaVu Sans", "DejaVu Serif", "DejaVu Sans Mono", "STIXGeneral"])),\n        "font.size": float(rr.uniform(7, 13)), "lines.linewidth": float(rr.uniform(0.7, 3.2)),\n        "lines.markersize": float(rr.uniform(2, 8)), "axes.linewidth": float(rr.uniform(0.4, 2.0)),\n        "axes.grid": bool(rr.random() < 0.5), "grid.alpha": float(rr.uniform(0.2, 0.9)),\n        "grid.linestyle": str(rr.choice(["-", "--", ":"])), "axes.facecolor": tint(),\n        "figure.facecolor": tint() if rr.random() < 0.3 else "white",\n        "axes.spines.top": bool(rr.random() < 0.5), "axes.spines.right": bool(rr.random() < 0.5),\n        "xtick.direction": str(rr.choice(["in", "out"])), "ytick.direction": str(rr.choice(["in", "out"])),\n    }\n\n\ndef _seed(sid):\n    # crc32, not hash(): str hashing is salted per process, so styles were not reproducible\n    return zlib.crc32(sid.encode()) % (2 ** 31)\n\n\ndef _px(fig, ax, xd, yd):\n    """Data coords -> image pixel coords (origin top-left)."""\n    pts = ax.transData.transform(np.column_stack([np.atleast_1d(xd), np.atleast_1d(yd)]))\n    return pts[:, 0], H_PX - pts[:, 1]\n\n\ndef _box_from_data(fig, ax, x0, y0, x1, y1, label):\n    xs, ys = _px(fig, ax, [x0, x1], [y0, y1])\n    a, b = sorted([float(xs[0]), float(xs[1])])\n    c, d = sorted([float(ys[0]), float(ys[1])])\n    return {"label": label,\n            "bbox": [round(max(0, a), 1), round(max(0, c), 1),\n                     round(min(W_PX, b), 1), round(min(H_PX, d), 1)]}\n\n\ndef _box_from_artist(fig, artist, label):\n    try:\n        e = artist.get_window_extent(fig.canvas.get_renderer())\n    except Exception:\n        return None\n    return {"label": label,\n            "bbox": [round(max(0, float(e.x0)), 1), round(max(0, H_PX - float(e.y1)), 1),\n                     round(min(W_PX, float(e.x1)), 1), round(min(H_PX, H_PX - float(e.y0)), 1)]}\n\n\n# --------------------------------------------------------------------------\n# per-arity drawing. Each returns a list of (kind, args) evidence requests\n# resolved to pixel boxes after fig.canvas.draw().\n# --------------------------------------------------------------------------\ndef draw_univariate(ax, rec, r, colour):\n    st = rec["ground_truth"]["statistics"]\n    x = np.asarray(rec["ground_truth"]["data_table"]["x"], float)\n    ct = rec["chart_type"]\n    ev = []\n\n    if ct == "histogram":\n        nb = int(np.clip(np.sqrt(len(x)) * r.uniform(0.8, 1.4), 12, 45))\n        cnt, edges, _ = ax.hist(x, bins=nb, color=colour, edgecolor="white", alpha=0.88)\n        i = int(np.argmax(cnt))\n        ev.append(("data", edges[i], 0, edges[i + 1], cnt[i], "modal_bin"))\n        ax.set_ylabel("Frequency")\n    elif ct == "kde":\n        from scipy.stats import gaussian_kde\n        gr = np.linspace(x.min(), x.max(), 240)\n        dens = gaussian_kde(x)(gr)\n        ax.fill_between(gr, dens, color=colour, alpha=0.45)\n        ax.plot(gr, dens, color=colour, lw=2)\n        ev.append(("data", gr[int(np.argmax(dens))], 0, gr[int(np.argmax(dens))], dens.max(), "peak"))\n        ax.set_ylabel("Density")\n    elif ct == "boxplot":\n        bp = ax.boxplot(x, vert=True, patch_artist=True, widths=0.5)\n        bp["boxes"][0].set_facecolor(colour)\n        bp["boxes"][0].set_alpha(0.8)\n        ev.append(("data", 0.75, st["q1"], 1.25, st["q3"], "iqr_box"))\n        ev.append(("data", 0.6, st["median"], 1.4, st["median"], "median_line"))\n        ax.set_ylabel(rec["axis_labels"]["x"])\n        ax.set_xticks([])\n    elif ct == "violin":\n        vp = ax.violinplot(x, showmedians=True, widths=0.7)\n        for b in vp["bodies"]:\n            b.set_facecolor(colour)\n            b.set_alpha(0.7)\n        ev.append(("data", 0.7, st["q1"], 1.3, st["q3"], "iqr_region"))\n        ax.set_ylabel(rec["axis_labels"]["x"])\n        ax.set_xticks([])\n    elif ct == "ecdf":\n        s = np.sort(x)\n        ax.step(s, np.arange(1, len(s) + 1) / len(s), color=colour, lw=2)\n        ev.append(("data", st["median"], 0, st["median"], 0.5, "median_crossing"))\n        ax.set_ylabel("Cumulative proportion")\n    elif ct == "qq":\n        from scipy import stats as sps\n        (osm, osr), (sl, ic, _) = sps.probplot(x, dist="norm")\n        ax.scatter(osm, osr, s=14, color=colour, alpha=0.75)\n        ax.plot(osm, sl * osm + ic, "r--", lw=1.8)\n        ev.append(("data", osm.min(), (sl * osm + ic).min(), osm.max(), (sl * osm + ic).max(),\n                   "normal_reference_line"))\n        ax.set_xlabel("Theoretical quantiles")\n        ax.set_ylabel("Sample quantiles")\n    elif ct == "boxen":\n        import seaborn as sns\n        sns.boxenplot(y=x, color=colour, ax=ax)\n        ev.append(("data", -0.4, st["q1"], 0.4, st["q3"], "iqr_box"))\n        ax.set_ylabel(rec["axis_labels"]["x"])\n    else:  # strip\n        jitter = r.normal(0, 0.045, len(x))\n        ax.scatter(jitter, x, s=15, color=colour, alpha=0.55)\n        ax.set_xlim(-0.35, 0.35)\n        ev.append(("data", -0.3, st["q1"], 0.3, st["q3"], "iqr_region"))\n        ax.set_ylabel(rec["axis_labels"]["x"])\n        ax.set_xticks([])\n\n    if ct in ("histogram", "kde", "ecdf"):\n        ax.set_xlabel(rec["axis_labels"]["x"])\n    lo, hi = st.get("iqr_lower_fence"), st.get("iqr_upper_fence")\n    if lo is not None and st.get("outlier_count", 0) > 0:\n        if ct in ("histogram", "kde", "ecdf"):\n            if x.max() > hi:\n                ev.append(("data", hi, ax.get_ylim()[0], x.max(), ax.get_ylim()[1], "upper_outliers"))\n            if x.min() < lo:\n                ev.append(("data", x.min(), ax.get_ylim()[0], lo, ax.get_ylim()[1], "lower_outliers"))\n    return ev\n\n\ndef draw_bivariate(ax, rec, r, colour):\n    st = rec["ground_truth"]["statistics"]\n    d = rec["ground_truth"]["data_table"]\n    x, y = np.asarray(d["x"], float), np.asarray(d["y"], float)\n    ct = rec["chart_type"]\n    ev = []\n\n    if ct == "hexbin":\n        ax.hexbin(x, y, gridsize=int(r.integers(16, 28)), cmap="viridis", mincnt=1)\n    elif ct == "bubble":\n        sz = np.abs(r.normal(45, 18, len(x))).clip(8, 130)\n        ax.scatter(x, y, s=sz, color=colour, alpha=0.5, edgecolor="white", lw=0.5)\n    else:\n        ax.scatter(x, y, s=int(r.integers(14, 30)), color=colour, alpha=0.65,\n                   edgecolor="white", lw=0.4)\n\n    if ct == "regplot" or (ct == "scatter" and r.random() < 0.45):\n        b1, b0 = st.get("linear_slope"), st.get("linear_intercept")\n        if b1 is not None:\n            gx = np.linspace(x.min(), x.max(), 60)\n            ax.plot(gx, b1 * gx + b0, "r-", lw=2, alpha=0.9)\n            ev.append(("data", gx[0], b1 * gx[0] + b0, gx[-1], b1 * gx[-1] + b0, "fitted_trend"))\n\n    cls = rec["label"]\n    if cls == "rel_heteroscedastic":\n        q = np.quantile(x, [2 / 3, 1.0])\n        ev.append(("data", q[0], y.min(), q[1], y.max(), "high_variance_region"))\n        q2 = np.quantile(x, [0.0, 1 / 3])\n        ev.append(("data", q2[0], y.min(), q2[1], y.max(), "low_variance_region"))\n    elif cls == "rel_nonlinear_nonmonotonic":\n        c2 = st.get("curvature_coef") or 0\n        xs = (x - x.mean()) / (x.std() + 1e-12)\n        vi = int(np.argmin(xs ** 2))\n        ev.append(("data", x[vi] - 0.1 * x.std(), y.min(), x[vi] + 0.1 * x.std(), y.max(),\n                   "curvature_vertex" if c2 else "vertex"))\n\n    ax.set_xlabel(rec["axis_labels"]["x"])\n    ax.set_ylabel(rec["axis_labels"]["y"])\n    return ev\n\n\ndef draw_timeseries(ax, rec, r, colour):\n    st = rec["ground_truth"]["statistics"]\n    d = rec["ground_truth"]["data_table"]\n    t, y = np.asarray(d["t"], float), np.asarray(d["y"], float)\n    ct = rec["chart_type"]\n    ev = []\n    # OpenML/vega dates are stored as epoch microseconds; plotted raw they give a\n    # "1.585 1e15" axis. Plot as dates. Everything below (fits, boxes) reads this\n    # same t, so evidence boxes stay in the axis\'s own coordinates.\n    is_date = t.size and np.nanmin(t) > 1e14\n    if is_date:\n        import matplotlib.dates as mdates\n        t = mdates.date2num(t.astype("datetime64[us]"))\n\n    if ct == "area":\n        ax.fill_between(t, y, np.min(y), color=colour, alpha=0.45)\n        ax.plot(t, y, color=colour, lw=1.8)\n    elif ct == "step":\n        ax.step(t, y, where="mid", color=colour, lw=1.8)\n    elif ct == "line_marker":\n        ax.plot(t, y, color=colour, lw=1.5, marker="o", ms=3.5, alpha=0.9)\n    else:\n        ax.plot(t, y, color=colour, lw=1.9)\n\n    cls = rec["label"]\n    if cls in ("ts_increasing", "ts_decreasing"):\n        b1, b0 = np.polyfit(np.arange(len(y)), y, 1)\n        fit = b1 * np.arange(len(y)) + b0\n        if not CLEAN:\n            ax.plot(t, fit, "r--", lw=1.6, alpha=0.85)\n        ev.append(("data", t[0], fit[0], t[-1], fit[-1], "fitted_trend"))\n    elif cls == "ts_changepoint":\n        ci = st.get("changepoint_index")\n        if ci is not None and 0 < ci < len(t):\n            if not CLEAN:\n                ax.axvline(t[ci], color="red", ls="--", lw=1.5, alpha=0.8)\n            span = max(1, len(t) // 25)\n            lo = t[max(0, ci - span)]\n            hi = t[min(len(t) - 1, ci + span)]\n            ev.append(("data", lo, y.min(), hi, y.max(), "changepoint"))\n            ev.append(("data", t[0], y[:ci].mean(), t[ci], y[:ci].mean(), "level_before"))\n            ev.append(("data", t[ci], y[ci:].mean(), t[-1], y[ci:].mean(), "level_after"))\n    elif cls == "ts_anomaly_spike":\n        for i in (st.get("anomaly_indices") or [])[:6]:\n            if 0 <= i < len(t):\n                if not CLEAN:\n                    ax.scatter([t[i]], [y[i]], s=110, facecolor="none", edgecolor="red", lw=2,\n                               zorder=5)\n                w = (t[-1] - t[0]) / max(len(t), 1) * 2.5\n                h = (y.max() - y.min()) * 0.06\n                ev.append(("data", t[i] - w, y[i] - h, t[i] + w, y[i] + h, "anomaly_point"))\n    elif cls == "ts_seasonal":\n        p = st.get("seasonal_period")\n        if p and p > 1:\n            step = (t[-1] - t[0]) / max(len(t) - 1, 1)\n            ev.append(("data", t[0], y.min(), t[0] + p * step, y.max(), "one_seasonal_cycle"))\n\n    if is_date:\n        loc = mdates.AutoDateLocator(maxticks=7)\n        ax.xaxis.set_major_locator(loc)\n        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))\n    ax.set_xlabel(rec["axis_labels"]["x"])\n    ax.set_ylabel(rec["axis_labels"]["y"])\n    return ev\n\n\ndef draw_grouped(ax, rec, r, palette):\n    import seaborn as sns\n    st = rec["ground_truth"]["statistics"]\n    d = rec["ground_truth"]["data_table"]\n    g = np.asarray(d["group"])\n    v = np.asarray(d["value"], float)\n    order = sorted(set(g.tolist()))\n    ct = rec["chart_type"]\n    ev = []\n\n    if ct == "grouped_violin":\n        sns.violinplot(x=g, y=v, order=order, hue=g, hue_order=order, legend=False,\n                       palette=palette, ax=ax)\n    elif ct == "grouped_bar":\n        sns.barplot(x=g, y=v, order=order, hue=g, hue_order=order, legend=False,\n                    palette=palette, errorbar="sd", ax=ax)\n    elif ct == "pointplot":\n        sns.pointplot(x=g, y=v, order=order, color="tab:blue", errorbar="sd", ax=ax)\n    elif ct == "grouped_strip":\n        sns.stripplot(x=g, y=v, order=order, hue=g, hue_order=order, legend=False,\n                      palette=palette, alpha=0.6, ax=ax)\n    else:\n        sns.boxplot(x=g, y=v, order=order, hue=g, hue_order=order, legend=False,\n                    palette=palette, ax=ax)\n\n    meds = st.get("group_medians") or {}\n    for i, lv in enumerate(order):\n        sub = v[g == lv]\n        if len(sub):\n            ev.append(("data", i - 0.4, np.percentile(sub, 25), i + 0.4, np.percentile(sub, 75),\n                       f"group:{lv}"))\n    hi, lo = st.get("highest_group"), st.get("lowest_group")\n    for nm, tag in ((hi, "highest_median_group"), (lo, "lowest_median_group")):\n        if nm in order:\n            i = order.index(nm)\n            ev.append(("data", i - 0.45, v.min(), i + 0.45, v.max(), tag))\n\n    ax.set_xlabel(rec["axis_labels"]["x"])\n    ax.set_ylabel(rec["axis_labels"]["y"])\n    if max(len(str(o)) for o in order) > 8:\n        ax.tick_params(axis="x", rotation=20)\n    return ev\n\n\ndef draw_composition(ax, rec, r, palette):\n    st = rec["ground_truth"]["statistics"]\n    labs = np.asarray(rec["ground_truth"]["data_table"]["category"])\n    counts = st.get("counts") or {}\n    cats = list(counts)\n    vals = [counts[c] for c in cats]\n    ct = rec["chart_type"]\n    ev = []\n    cols = plt.get_cmap(palette if palette in plt.colormaps() else "tab10")(\n        np.linspace(0, 0.9, len(cats)))\n\n    if ct == "pie":\n        wedges, *_ = ax.pie(vals, labels=cats, autopct="%1.1f%%", colors=cols,\n                            startangle=int(r.integers(0, 90)))\n        ax.set_ylabel("")\n        return [("artist", wedges[0], "dominant_wedge")] + \\\n               [("artist", w, f"wedge:{c}") for w, c in list(zip(wedges, cats))[1:4]]\n    if ct == "barh":\n        bars = ax.barh(cats, vals, color=cols)\n        ax.set_xlabel("Count")\n        ax.set_ylabel(rec["axis_labels"]["x"])\n    else:\n        bars = ax.bar(cats, vals, color=cols)\n        ax.set_ylabel("Count")\n        ax.set_xlabel(rec["axis_labels"]["x"])\n        if max(len(str(c)) for c in cats) > 8:\n            ax.tick_params(axis="x", rotation=20)\n\n    out = [("artist", bars[0], "dominant_category_bar")]\n    out += [("artist", b, f"bar:{c}") for b, c in list(zip(bars, cats))[1:5]]\n    return out\n\n\nDRAW = {"univariate": draw_univariate, "bivariate": draw_bivariate,\n        "timeseries": draw_timeseries, "grouped": draw_grouped,\n        "composition": draw_composition}\n\n\ndef render_one(rec):\n    sid = rec["sample_id"]\n    r = np.random.default_rng(_seed(sid))\n    style, palette = STYLES[_seed(sid) % len(STYLES)]\n    out = IMG / rec["label"] / f"{sid}.png"\n    out.parent.mkdir(parents=True, exist_ok=True)\n    rr = np.random.default_rng(_seed(sid + "|rand")) if RANDSTYLE else None\n    rc = _rand_rc(rr) if RANDSTYLE else {}\n    if RANDSTYLE:\n        palette = str(rr.choice(RAND_CMAPS))\n\n    try:\n        with plt.style.context(style), plt.rc_context(rc):\n            fig = plt.figure(figsize=FIGSIZE, dpi=DPI)\n            if rec["chart_type"] == "jointplot":\n                # built by hand rather than seaborn.jointplot, which owns its own\n                # figure and would break the fixed canvas the boxes depend on\n                gs = GridSpec(4, 4, figure=fig, hspace=0.05, wspace=0.05)\n                ax = fig.add_subplot(gs[1:, :3])\n                axt = fig.add_subplot(gs[0, :3], sharex=ax)\n                axr = fig.add_subplot(gs[1:, 3], sharey=ax)\n                d = rec["ground_truth"]["data_table"]\n                axt.hist(np.asarray(d["x"], float), bins=28, color="tab:blue", alpha=0.7)\n                axr.hist(np.asarray(d["y"], float), bins=28, orientation="horizontal",\n                         color="tab:blue", alpha=0.7)\n                axt.axis("off")\n                axr.axis("off")\n            else:\n                ax = fig.add_subplot(111)\n\n            cmap = plt.get_cmap(palette if palette in plt.colormaps() else "tab10")\n            colour = cmap(int(r.integers(0, 8)) % cmap.N)\n            if RANDSTYLE:\n                import colorsys\n                colour = colorsys.hsv_to_rgb(rr.uniform(0, 1), rr.uniform(0.35, 1.0), rr.uniform(0.3, 0.85))\n            fn = DRAW[rec["arity"]]\n            reqs = fn(ax, rec, r, palette if rec["arity"] in ("grouped", "composition") else colour)\n\n            title = rec["axis_labels"].get("y") or rec["axis_labels"].get("x")\n            if rec["arity"] == "univariate":\n                title = f"Distribution of {rec[\'axis_labels\'][\'x\']}"\n            elif rec["arity"] == "bivariate":\n                title = f"{rec[\'axis_labels\'][\'y\']} vs {rec[\'axis_labels\'][\'x\']}"\n            elif rec["arity"] == "timeseries":\n                title = f"{rec[\'axis_labels\'][\'y\']} over {rec[\'axis_labels\'][\'x\']}"\n            elif rec["arity"] == "grouped":\n                title = f"{rec[\'axis_labels\'][\'y\']} by {rec[\'axis_labels\'][\'x\']}"\n            else:\n                title = f"Composition of {rec[\'axis_labels\'][\'x\']}"\n            if not RANDSTYLE:\n                ax.set_title(title[:78], fontsize=11, fontweight="bold")\n            elif rr.random() > 0.3:\n                ax.set_title(title[:78], fontsize=float(rr.uniform(8, 14)),\n                             fontweight=str(rr.choice(["bold", "normal"])), loc=str(rr.choice(["center", "left"])))\n\n            fig.subplots_adjust(left=0.13, right=0.965, top=0.90, bottom=0.13)\n            fig.canvas.draw()\n\n            boxes = [{"label": "plot_area",\n                      "bbox": [round(float(ax.bbox.x0), 1), round(float(H_PX - ax.bbox.y1), 1),\n                               round(float(ax.bbox.x1), 1), round(float(H_PX - ax.bbox.y0), 1)]}]\n            for req in reqs or []:\n                if req[0] == "data":\n                    _, x0, y0, x1, y1, lab = req\n                    try:\n                        boxes.append(_box_from_data(fig, ax, x0, y0, x1, y1, lab))\n                    except Exception:\n                        pass\n                else:\n                    b = _box_from_artist(fig, req[1], req[2])\n                    if b:\n                        boxes.append(b)\n\n            fig.savefig(out, dpi=DPI, facecolor="white")\n            plt.close(fig)\n    except Exception as e:\n        plt.close("all")\n        return sid, None, f"{type(e).__name__}: {e}"\n\n    return sid, {"image_size": [W_PX, H_PX], "style": style,\n                 "grounding_boxes": boxes[:14]}, None\n\n\ndef _work(chunk):\n    return [render_one(r) for r in chunk]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--clean", action="store_true", help="no label-dependent marks")\n    ap.add_argument("--randstyle", action="store_true", help="training-only style randomisation")\n    ap.add_argument("--split", help="render only this split")\n    ap.add_argument("--arity", help="render only this arity (e.g. timeseries)")\n    ap.add_argument("--records", default=str(CACHE / "records.jsonl"),\n                    help="records to render (default cache/records.jsonl)")\n    ap.add_argument("--out", help="image dir relative to dataset/ (default images); "\n                                  "records then go to <out>/records.jsonl")\n    a = ap.parse_args()\n    global IMG, CLEAN\n    if a.out:\n        os.environ["RENDER_OUT"] = str(DATA / a.out)\n        IMG = DATA / a.out\n    if a.clean:\n        os.environ["RENDER_CLEAN"] = "1"\n        CLEAN = True\n    if a.randstyle:\n        os.environ["RENDER_RANDSTYLE"] = "1"\n\n    recs = [json.loads(l) for l in open(a.records)]\n    recs = [r for r in recs if (not a.split or r["split"] == a.split)\n            and (not a.arity or r["arity"] == a.arity)]\n    print(f"rendering {len(recs):,} charts at {W_PX}x{H_PX} "\n          f"({len(set(r[\'chart_type\'] for r in recs))} chart types) clean={CLEAN} -> {IMG}")\n    IMG.mkdir(parents=True, exist_ok=True)\n\n    from concurrent.futures import ProcessPoolExecutor, as_completed\n    chunks = [recs[i:i + 40] for i in range(0, len(recs), 40)]\n    got, errs = {}, Counter()\n    with ProcessPoolExecutor(max_workers=max(1, os.cpu_count() - 2)) as ex:\n        futs = [ex.submit(_work, c) for c in chunks]\n        bar = tqdm(as_completed(futs), total=len(futs), unit="batch",\n                   desc="rendering charts", ncols=100)\n        done = 0\n        for fu in bar:\n            for sid, meta, err in fu.result():\n                done += 1\n                if meta:\n                    got[sid] = meta\n                else:\n                    errs[err.split(":")[0]] += 1\n            bar.set_postfix(ok=len(got), failed=sum(errs.values()))\n\n    out = IMG / "records.jsonl" if a.out else DATA / "records.jsonl"\n    kept = 0\n    with open(out, "w") as fh:\n        for r in recs:\n            m = got.get(r["sample_id"])\n            if not m:\n                continue\n            r["render"] = m\n            fh.write(json.dumps(r) + "\\n")\n            kept += 1\n\n    print(f"\\nrendered {kept:,}/{len(recs):,} charts -> {IMG}")\n    if errs:\n        print("failures by type:")\n        for k, v in errs.most_common(8):\n            print(f"  {k}: {v}")\n    print(f"records with render metadata -> {out}")\n\n\nif __name__ == "__main__":\n    main()\n'

import subprocess, sys
_w = Path("/tmp/v2render"); (_w / "build").mkdir(parents=True, exist_ok=True); (_w / "cache").mkdir(exist_ok=True)
(_w / "build" / "06_render.py").write_text(RENDER_SRC)
with open(DATA / "records.jsonl") as _f, open(_w / "cache" / "records.jsonl", "w") as _g:
    for _l in _f:
        if '"arity": "timeseries"' in _l: _g.write(_l)
OV = _w / "dataset" / "images_v2ts"
if not OV.exists():
    subprocess.run([sys.executable, str(_w / "build" / "06_render.py"), "--arity", "timeseries",
                    "--clean", "--out", "images_v2ts"], check=True)

def v2img(p):
    """Time-series charts come from the clean v2 render; every other chart is unchanged."""
    p = str(p)
    return str(OV / p.split("/", 1)[1]) if p.startswith("images/ts_") else str(DATA / p)
assert all(Path(v2img(p)).exists() for p in te.image), "v2 image missing"
print("v2 images:", sum(p.startswith("images/ts_") for p in te.image), "test time series from the clean render")

In [ ]:
from transformers import (AutoProcessor, Qwen2VLForConditionalGeneration,
                          BitsAndBytesConfig, TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
# 1500 keeps the run near 1.5 h on a T4. Raise it if quota allows -- the
# size is printed and stored in nb4_results.json, so report whatever you use.
N_TRAIN = 4000
rows = [json.loads(l) for l in open(DATA / "instruct" / "train.jsonl")][:N_TRAIN]
for r in rows: r["image"] = v2img(r["image"])
print(len(rows), "training samples (subset -- report this size in the paper)")

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)   # T4 is fp16, not bf16
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=768*28*28)
vlm = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
vlm = prepare_model_for_kbit_training(vlm); vlm.config.use_cache = False
vlm = get_peft_model(vlm, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
vlm.print_trainable_parameters()

In [ ]:
class VLMDataset(TorchDS):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    texts, images = [], []
    for b in batch:
        msgs = [{"role":"user","content":[{"type":"image","image":b["image"]},
                 {"type":"text","text":b["messages"][0]["content"].replace("<image>\n","")}]},
                {"role":"assistant","content":[{"type":"text","text":b["messages"][1]["content"]}]}]
        texts.append(processor.apply_chat_template(msgs, tokenize=False))
        imgs, _ = process_vision_info(msgs); images.append(imgs)
    enc = processor(text=texts, images=images, return_tensors="pt",
                    padding=True, truncation=True, max_length=1024)
    labels = enc["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    for tok in ("<|image_pad|>","<|vision_start|>","<|vision_end|>"):
        tid = processor.tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid >= 0: labels[labels == tid] = -100
    enc["labels"] = labels
    return enc


import inspect
_want = dict(output_dir="insightvlm-lora", num_train_epochs=1,
             per_device_train_batch_size=1, gradient_accumulation_steps=8,
             learning_rate=1e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
             logging_steps=10, save_strategy="epoch", fp16=True,
             gradient_checkpointing=True, optim="paged_adamw_8bit",
             report_to="none", remove_unused_columns=False, dataloader_num_workers=2)
_sig = inspect.signature(TrainingArguments.__init__).parameters
_ok = {k: v for k, v in _want.items() if k in _sig}
_dropped = sorted(set(_want) - set(_ok))
print("transformers", __import__("transformers").__version__)
if _dropped:
    print("  unsupported in this build, dropped:", _dropped)
targs = TrainingArguments(**_ok)


from transformers import TrainerCallback
class PlainProgress(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if logs:
            el = time.time() - _t0
            done = state.global_step / max(state.max_steps, 1)
            eta = el / done - el if done > 0 else 0
            print(f"  step {state.global_step}/{state.max_steps} "
                  f"({100*done:.0f}%)  " +
                  "  ".join(f"{k}={v:.4g}" for k, v in logs.items() if isinstance(v, (int, float))) +
                  f"  elapsed {el/60:.0f}m  eta {eta/60:.0f}m", flush=True)

_t0 = time.time()
trainer = Trainer(model=vlm, args=targs, train_dataset=VLMDataset(rows),
                  data_collator=collate, callbacks=[PlainProgress()])
trainer.remove_callback(type(trainer.callback_handler.callbacks[-2]))  # drop the widget one
print(f"training {len(rows)} samples -> {targs.max_steps if targs.max_steps>0 else len(rows)//8} steps")
trainer.train()
vlm.save_pretrained("insightvlm-lora/final")
print("saved -> insightvlm-lora/final")

In [ ]:
vlm.eval()
PROMPT = ("You are a data analyst. Examine this chart and report the statistical "
          "findings it supports. State the dominant pattern, cite the numbers that "
          "justify it, and note any caveat a reader should know.")

def generate(path, max_new_tokens=320):
    msgs = [{"role":"user","content":[{"type":"image","image":path},{"type":"text","text":PROMPT}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs, _ = process_vision_info(msgs)
    inp = processor(text=[text], images=[imgs], return_tensors="pt").to(vlm.device)
    with torch.no_grad():
        out = vlm.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0]

CLS = json.loads((DATA/"class_index.json").read_text())["classes"]
DESC = {CLS[c]["desc"]: c for c in CLASSES}
def parse_class(t):
    for d,c in DESC.items():
        if d[:40].lower() in t.lower(): return c
    for c in CLASSES:
        if c.replace("_"," ") in t.lower(): return c
    return None


ev = te.sample(300, random_state=0); out_rows = []
for k,(_,r) in enumerate(ev.iterrows()):
    txt = generate(v2img(r.image))
    gt = set(re.findall(r"-?\d+\.\d{2}", str(json.loads(r.statistics_json))))
    pr = set(re.findall(r"-?\d+\.\d{2}", txt))
    out_rows.append({"true":r.label,"pred":parse_class(txt),"provenance":r.provenance,
                     "numeric_fidelity":len(gt&pr)/max(len(pr),1),"text":txt})
    if k % 25 == 0: print(f"\r  {k}/{len(ev)}", end="", flush=True)

v = pd.DataFrame(out_rows); v["ok"] = v.true == v.pred
print(f"\n\nVLM class accuracy   {v.ok.mean():.4f}")
print(f"VLM numeric fidelity {v.numeric_fidelity.mean():.4f}")
display(v.groupby("provenance").ok.agg(["mean","count"]).round(4))
print("\nA large real-vs-synthetic gap means template recitation, not chart reading.")
v.to_csv("nb4_vlm_predictions.csv", index=False)
Path("nb4_results.json").write_text(json.dumps({
    "n_train":len(rows), "n_eval":len(v), "class_accuracy":float(v.ok.mean()),
    "numeric_fidelity":float(v.numeric_fidelity.mean()),
    "by_provenance":v.groupby("provenance").ok.mean().round(4).to_dict()}, indent=2))
print("wrote nb4_results.json")